In [1]:
from decouple import AutoConfig
config = AutoConfig(search_path='./../.env')

In [2]:
import os
import openai

openai.api_key = config('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai.api_key

### Defining the LLM 

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gpt-4o-mini'
llm = ChatOpenAI(
    model=model_name,
    temperature=0.0,
    max_tokens=1024
)

llm

### Defining the search tools

In [4]:
from langchain_community.tools.arxiv.tool import ArxivQueryRun
from langchain_community.tools.tavily_search import TavilySearchResults

arxiv_search = ArxivQueryRun()
tavily_tool = TavilySearchResults(max_results=5)

tools = [arxiv_search, tavily_tool]


### Defining the Graph state

In [5]:
from typing import TypedDict, Annotated
from IPython.display import Image, display

In [6]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
class AgentState(TypedDict):
    input: str
    messages: Annotated[list[AnyMessage], add_messages]

`add_messages` is a reducer function that appends messages to `messages` state key.

### Initialising the workflow(graph)

In [17]:
from langgraph.graph import StateGraph
workflow = StateGraph(AgentState)

### Defining the agent (node) 

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

def research_agent(data):
    print("----research node----")
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful AI research assistant,"
                " Use the appropriate search tools to progress towards finding the relevant results."
                " Once you have the relevant search results, summarise them to answer the user query."
                "\nYou have access to the following search tools: {tool_names}."
            ),
            (
                "human",
                "\nUser Query: {input}"
            ),
            
            MessagesPlaceholder(variable_name="messages"),
        ]
    )
    prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))
    agent = prompt | llm.bind_tools(tools)
    result = agent.invoke(data)
    return {'messages': [result]}

#### Adding researcher agent to the workflow

In [ ]:
workflow.add_node("research", research_agent)

#### Setting researcher agent as the entry point of the workflow

In [ ]:
workflow.set_entry_point("research")

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition
## Initialising tools as tool node.
## This instantiate tool node with the list of search tools.
tools_node = ToolNode(tools)

## Adding tools_node to the workflow
workflow.add_node("tools", tools_node)

`ToolNode` takes list of tools and initialise it.

In [ ]:
## Adding Conditional edge

workflow.add_conditional_edges(
    "research",
    tools_condition
)

`tools_condition` is a built-in conditional edge for routing tool calls.

In [ ]:
## Adding edge back from tools to researcher

workflow.add_edge("tools", "research")

**NOTE:** To use `langgraph.prebuilt.ToolNode` and `langgraph.prebuilt.tools_conditions` refer langgraph ToolNode [documentation](https://langchain-ai.github.io/langgraph/reference/prebuilt/). For custom applications, I prefer defining `ToolNode` and `route_tools` based on the application's requirements.

In [ ]:
app = workflow.compile()
try:
    display(Image(app.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [ ]:
inputs = {
    "input": "What are the recent papers on Small Language Models?",
}

app.invoke(inputs)

In [ ]:
state = AgentState(**inputs)
for s in app.stream(input=state):
    print(list(s.values())[0]['messages'][0].content)
    print("-----"*20)